In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente.")

In [ ]:
# Ruta a la carpeta que contiene los CSVs
folder_path = 'partidos'

# Lista para almacenar los DataFrames de cada partido
dfs = []

print(f"Extrayendo y consolidando CSVs de la carpeta '{folder_path}'...")

# Leer cada archivo CSV de la carpeta
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

for file in csv_files:
    file_path = os.path.join(folder_path, file)
    # Cargar en pandas con el encoding adecuado para caracteres en español
    df_partido = pd.read_csv(file_path, encoding='ISO-8859-1', sep=',')
    dfs.append(df_partido)

# Unir todos los partidos en un único DataFrame maestro
df_master = pd.concat(dfs, ignore_index=True)

print(f"Consolidación completada. Total de registros (eventos): {df_master.shape[0]}")
print(f"Total de columnas: {df_master.shape[1]}")

In [ ]:
# Limpiar el nombre de la primera columna que contiene caracteres de formato extraños
df_master.rename(columns={'ï»¿id_competition': 'id_competition'}, inplace=True)

# Convertir las columnas de tiempo y coordenadas a numéricas
cols_numericas = ['posX', 'posY', 'minute', 'second', 'score_local', 'score_visitor']
for col in cols_numericas:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce')

df_master.info()

In [ ]:
# Crear una columna unificada para identificar el tipo de tiro
condiciones_tiro = [
    df_master['action_desc'].str.contains('Canasta de 3|Intento fallado de 3', case=False, na=False),
    df_master['action_desc'].str.contains('Canasta de 2|Intento fallado de 2|Mate', case=False, na=False),
    df_master['action_desc'].str.contains('Canasta de 1|Intento fallado de 1', case=False, na=False)
]
valores_tiro = ['3PT', '2PT', 'FT'] 

df_master['shot_type'] = np.select(condiciones_tiro, valores_tiro, default=None)

# Crear variable booleana para tiros anotados (Shot Made = 1, Missed = 0)
df_master['shot_made'] = df_master['action_desc'].str.contains('Canasta|Mate', case=False, na=False).astype(float)

# Si no es un tiro (shot_type es None), shot_made debe ser NaN
df_master.loc[df_master['shot_type'].isna(), 'shot_made'] = np.nan

print("Variables de análisis de tiro creadas con éxito.")
df_master[['action_desc', 'shot_type', 'shot_made']].dropna().head()

In [ ]:
# Guardamos el dataset limpio para usarlo en el análisis espacial
output_file = 'acb_master_playbyplay.csv'
df_master.to_csv(output_file, index=False, encoding='utf-8')
print(f"Dataset maestro exportado con éxito a: {output_file}")